In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm

from data_loader import build_complete_dataset
from window import create_record_windows


In [2]:
def extract_window_features(window_df, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"{prefix}_{col}_mean"] = np.mean(signal)
            features[f"{prefix}_{col}_std"] = np.std(signal)
            features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"{prefix}_{col}_max"] = np.max(signal)
            features[f"{prefix}_{col}_min"] = np.min(signal)
            features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
            features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals**2)
            features[f"{prefix}_{col}_fft_peak_freq"] = np.argmax(fft_vals)
    return features

In [3]:
def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)
    
    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))
    
    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], prefix="mic"))
        
        features["segment_id"] = record.metadata.get("segment_id")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)
        
    return pd.DataFrame(record_features)

In [4]:
def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [5]:
df = build_complete_dataset()

X_train_df = build_feature_dataset(df[0], create_record_windows)

X_test_df = build_feature_dataset(df[1], create_record_windows)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())


Ekstrakcja cech: 100%|██████████| 464/464 [05:20<00:00,  1.45it/s]

Train: (272172, 74)
Test : (68672, 74)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [6]:
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

acc_gyro_features = acc_features + gyro_features
acc_mic_features = acc_features + mic_features
acc_gyro_mic_features = acc_features + gyro_features + mic_features

In [7]:
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test_df[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test_df[mic_features].fillna(0)


X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test_df[gyro_features].fillna(0)

X_train_acc_mic = X_train_df[acc_mic_features].fillna(0)
X_test_acc_mic = X_test_df[acc_mic_features].fillna(0)

X_train_acc_gyro = X_train_df[acc_gyro_features].fillna(0)
X_test_acc_gyro = X_test_df[acc_gyro_features].fillna(0)

X_train_gyro_mic = X_train_df[gyro_features + mic_features].fillna(0)
X_test_gyro_mic = X_test_df[gyro_features + mic_features].fillna(0)

X_train_acc_gyro_mic = X_train_df[acc_gyro_mic_features].fillna(0)
X_test_acc_gyro_mic = X_test_df[acc_gyro_mic_features].fillna(0)

In [10]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)


def run_isolation_forest(X_train, X_test, train_df, test_df, sensor_type):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = IsolationForest(
        n_estimators=300,
        contamination=0.01,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_scaled)

    scores = -model.score_samples(X_test_scaled)

    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
        "anomaly_label": test_df["anomaly_label"].values,
        "score": scores})

    segment_scores = (results.groupby("segment_id").agg({"score": "mean", "anomaly_label": "first"}).reset_index())

    y_true = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)

    threshold = np.percentile(segment_scores["score"], 50)

    y_pred = (segment_scores["score"] >= threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred,zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, segment_scores["score"])
    cm = confusion_matrix(y_true, y_pred)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    return {"Sensors": sensor_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]}

In [ ]:
results = []

print("ACC")
result_acc = run_isolation_forest(X_train_acc, X_test_acc, X_train_df, X_test_df, "ACC")
results.append(result_acc)

print("MIC")
result_mic = run_isolation_forest(X_train_mic, X_test_mic, X_train_df, X_test_df, "MIC")
results.append(result_mic)

print("GYRO")
result_gyro = run_isolation_forest(X_train_gyro, X_test_gyro, X_train_df, X_test_df, "GYRO")
results.append(result_gyro)

print("ACC + GYRO")
result_acc_gyro = run_isolation_forest(X_train_acc_gyro, X_test_acc_gyro, X_train_df, X_test_df, "ACC + GYRO")
results.append(result_acc_gyro)


print("ACC + MIC")
result_acc_mic = run_isolation_forest(X_train_acc_mic, X_test_acc_mic, X_train_df, X_test_df, "ACC + MIC")
results.append(result_acc_mic)

print("GYRO + MIC")
result_gyro_mic = run_isolation_forest(X_train_gyro_mic, X_test_gyro_mic, X_train_df, X_test_df, "GYRO + MIC")
results.append(result_gyro_mic)

print("ACC + GYRO + MIC")
result_acc_gyro_mic = run_isolation_forest(X_train_acc_gyro_mic, X_test_acc_gyro_mic, X_train_df, X_test_df, "ACC + GYRO + MIC")
results.append(result_acc_gyro_mic)


ACC
Accuracy : 0.6121
Precision: 0.6121
Recall   : 0.6121
F1-score : 0.6121
ROC AUC  : 0.5443
[[142  90]
 [ 90 142]]
MIC
Accuracy : 0.5603
Precision: 0.5603
Recall   : 0.5603
F1-score : 0.5603
ROC AUC  : 0.5923
[[130 102]
 [102 130]]
GYRO
